## 2 序列模型

### 2.1 理论计算题

字符序列："ababc"，词汇表 {a, b, c}

统计一阶转移频次：
- a→b 出现 2 次
- b→a 出现 2 次
- b→c 出现 1 次

转移总次数 = 5（序列长度6，相邻对5个）

使用加1平滑公式：

$$p(x'|y') = \frac{count(y' \to x') + 1}{count(y' \to \cdot) + |V|}$$

**1. p(a|b)**

count(b→a) = 2，count(b→·) = 3，|V| = 3

$$p(a|b) = \frac{2 + 1}{3 + 3} = \frac{3}{6} = 0.5$$

**2. p(c|b)**

count(b→c) = 1，count(b→·) = 3，|V| = 3

$$p(c|b) = \frac{1 + 1}{3 + 3} = \frac{2}{6} = \frac{1}{3} \approx 0.3333$$

**答案：** p(a|b) = 0.5，p(c|b) = 1/3

### 2.2 编程题

做了什么操作： 定义了一个 preprocess_text 函数，先将文本转小写并去除非字母和空格的字符，然后按空格分词，用 Counter 按出现频率排序构建词汇表并分配整数 ID，最后用滑动窗口生成长度为 n 的特征序列和对应的下一个词标签。测试时输入 "The time machine" 和 n=2，打印了词汇表、特征和标签。

In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    # 1. 转小写，去除非字母和空格的字符
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 2. 按空格分词
    words = text.split()
    
    # 3. 构建词汇表（按出现频率排序）
    word_counts = Counter(words)
    vocab = {word: idx for idx, (word, _) in enumerate(word_counts.most_common())}
    
    # 4. 生成滑动窗口序列
    features = []
    labels = []
    for i in range(len(words) - n):
        features.append(words[i:i+n])
        labels.append(words[i+n])
    
    return vocab, (features, labels)


# 测试
vocab, (features, labels) = preprocess_text("The time machine", 2)
print("词汇表:", vocab)
print("特征:", features)
print("标签:", labels)

词汇表: {'the': 0, 'time': 1, 'machine': 2}
特征: [['the', 'time']]
标签: ['machine']


## 3 循环神经网络

### 3.1 理论计算题

RNN 定义：$h_t = W_{hh}h_{t-1} + W_{hx}x_t$，$o_t = W_{oh}h_t$

损失：$L = \frac{1}{2}\sum_{t=1}^{T}(o_t - y_t)^2$

对 $W_{hh}$ 的梯度为所有时间步贡献之和：

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \frac{\partial L_t}{\partial W_{hh}}$$

对于时间步 $t$，损失 $L_t = \frac{1}{2}(W_{oh}h_t - y_t)^2$

根据链式法则：

$$\frac{\partial L_t}{\partial W_{hh}} = \frac{\partial L_t}{\partial h_t} \cdot \frac{\partial h_t}{\partial W_{hh}}$$

展开 $h_t$ 对 $W_{hh}$ 的依赖，需要回溯到所有之前的时间步：

$$h_t = W_{hh}h_{t-1} + W_{hx}x_t = W_{hh}(W_{hh}h_{t-2} + W_{hx}x_{t-1}) + W_{hx}x_t = \cdots$$

$$\frac{\partial h_t}{\partial W_{hh}} = \sum_{k=1}^{t} \left( \prod_{j=k+1}^{t} W_{hh} \right) h_{k-1}$$

因此：

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \frac{\partial L_t}{\partial h_t} \sum_{k=1}^{t} \left( \prod_{j=k+1}^{t} W_{hh} \right) h_{k-1}$$

**梯度消失/爆炸条件：**

- 当 $\|W_{hh}\| < 1$ 时，乘积项 $\prod_{j=k+1}^{t} W_{hh}$ 随 $t-k$ 增大趋于 0 → **梯度消失**
- 当 $\|W_{hh}\| > 1$ 时，乘积项随 $t-k$ 增大指数增长 → **梯度爆炸**

### 3.2 编程题

做了什么操作： 实现了 RNN 单元的前向传播和单步反向传播。前向传播用 tanh 激活函数计算当前隐藏状态。反向传播已知上游梯度 dh_next，先计算 tanh 的导数，然后分别计算 dW_hh、dW_xh、db_h、dx_t、dh_prev。测试时随机初始化了输入、隐藏状态和权重，打印了各梯度的形状。

In [4]:
import numpy as np

def rnn_cell_forward(x_t, h_prev, W_hh, W_xh, b_h):
    """
    前向传播：计算当前隐藏状态
    x_t: (batch_size, input_size)
    h_prev: (batch_size, hidden_size)
    W_hh: (hidden_size, hidden_size)
    W_xh: (hidden_size, input_size)
    b_h: (hidden_size, 1)
    返回: h_t (batch_size, hidden_size)
    """
    # 修正：h_prev (4,5) 点乘 W_hh (5,5) => (4,5)，符合要求
    h_t = np.tanh(np.dot(h_prev, W_hh) + np.dot(x_t, W_xh.T) + b_h.T)
    return h_t


def rnn_cell_backward(x_t, h_prev, h_t, dh_next, W_hh, W_xh, b_h):
    """
    反向传播：计算梯度
    dh_next: (batch_size, hidden_size) 损失对h_t的梯度
    返回: dx_t, dh_prev, dW_hh, dW_xh, db_h
    """
    # tanh 的导数: 1 - tanh^2
    dh = dh_next * (1 - h_t**2)
    
    # 参数的梯度
    dW_hh = np.dot(h_prev.T, dh)
    dW_xh = np.dot(x_t.T, dh)
    db_h = np.sum(dh, axis=0, keepdims=True).T
    
    # 对输入的梯度
    dx_t = np.dot(dh, W_xh)
    dh_prev = np.dot(dh, W_hh.T)
    
    return dx_t, dh_prev, dW_hh, dW_xh, db_h


# 测试
batch_size, input_size, hidden_size = 4, 3, 5
x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hh = np.random.randn(hidden_size, hidden_size)
W_xh = np.random.randn(hidden_size, input_size)
b_h = np.random.randn(hidden_size, 1)

h_t = rnn_cell_forward(x_t, h_prev, W_hh, W_xh, b_h)
dh_next = np.random.randn(batch_size, hidden_size)

dx_t, dh_prev, dW_hh, dW_xh, db_h = rnn_cell_backward(
    x_t, h_prev, h_t, dh_next, W_hh, W_xh, b_h
)

print("前向输出 h_t 形状:", h_t.shape)
print("dx_t 形状:", dx_t.shape)
print("dh_prev 形状:", dh_prev.shape)
print("dW_hh 形状:", dW_hh.shape)
print("dW_xh 形状:", dW_xh.shape)
print("db_h 形状:", db_h.shape)

前向输出 h_t 形状: (4, 5)
dx_t 形状: (4, 3)
dh_prev 形状: (4, 5)
dW_hh 形状: (5, 5)
dW_xh 形状: (3, 5)
db_h 形状: (5, 1)


## 4 高级循环神经网络

### 4.1 理论计算题

深度双向 RNN：L 层，每层隐藏单元数 H，输入维度 D

**第一层前向 RNN：**
- 输入到隐藏：$W_{xh}^{(1)}$：D×H，偏置 $b_h^{(1)}$：H
- 隐藏到隐藏：$W_{hh}^{(1)}$：H×H，偏置：H
- 参数数量：$DH + H + H^2 + H = DH + H^2 + 2H$

**第一层反向 RNN：** 同样结构，参数数量相同：$DH + H^2 + 2H$

**中间层（第 l 层，l ≥ 2）：**
- 前向 RNN 输入来自上一层前向和反向的拼接，输入维度为 2H
- 输入到隐藏：$2H \times H$，偏置：H
- 隐藏到隐藏：$H \times H$，偏置：H
- 参数数量：$2H^2 + H + H^2 + H = 3H^2 + 2H$
- 反向 RNN 同样：$3H^2 + 2H$

**总参数量：**

第一层（前向+反向）：$2(DH + H^2 + 2H)$

其余 L-1 层（前向+反向）：$2(L-1)(3H^2 + 2H)$

$$2DH + 2H^2 + 4H + 2(L-1)(3H^2 + 2H)$$

### 4.2 编程题

做了什么操作： 定义了一个 BiRNNEncoder 类，用 torch.nn.RNN 实现双向编码器，设置 bidirectional=True。前向传播返回每个时间步拼接后的前向和后向隐藏状态，以及最终时间步的拼接隐藏状态作为序列表示。测试时输入形状为 (10, 4, 8)，打印了 outputs 和 final_state 的形状。

In [5]:
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super(BiRNNEncoder, self).__init__()
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=False
        )
    
    def forward(self, X):
        # X: (seq_len, batch, input_dim)
        outputs, h_n = self.rnn(X)
        # outputs: (seq_len, batch, 2*hidden_dim)
        
        # 最终时间步的拼接隐藏状态
        # h_n: (num_layers*2, batch, hidden_dim)
        # 取最后一层的前向和后向拼接
        final_forward = h_n[-2, :, :]   # 最后一层前向
        final_backward = h_n[-1, :, :]  # 最后一层后向
        final_state = torch.cat([final_forward, final_backward], dim=1)
        # final_state: (batch, 2*hidden_dim)
        
        return outputs, final_state


# 测试
seq_len, batch, input_dim, hidden_dim = 10, 4, 8, 16
encoder = BiRNNEncoder(input_dim, hidden_dim)
X = torch.randn(seq_len, batch, input_dim)

outputs, final_state = encoder(X)
print("outputs 形状:", outputs.shape)        # (10, 4, 32)
print("final_state 形状:", final_state.shape)  # (4, 32)

outputs 形状: torch.Size([10, 4, 32])
final_state 形状: torch.Size([4, 32])


## 5 嵌入向量

### 5.1 理论计算题

Skip-gram 负采样目标函数：

对于中心词 $w_c$ 和上下文词 $w_o$，正样本对为 $(w_c, w_o)$，负样本从噪声分布 $P_n(w)$ 中采样 K 个。

**目标函数（最大化）：**

$$\mathcal{L} = \log \sigma(\mathbf{v}_c \cdot \mathbf{u}_o) + \sum_{k=1}^{K} \mathbb{E}_{w_n \sim P_n(w)} \left[ \log \sigma(-\mathbf{v}_c \cdot \mathbf{u}_{n_k}) \right]$$

**等价的最小化损失函数：**

$$\mathcal{L} = -\log \sigma(\mathbf{v}_c \cdot \mathbf{u}_o) - \sum_{k=1}^{K} \log \sigma(-\mathbf{v}_c \cdot \mathbf{u}_{n_k})$$

其中 $\sigma(x) = \frac{1}{1+e^{-x}}$

**噪声分布采样：**

常用噪声分布为 unigram 分布的 $3/4$ 次方：

$$P_n(w) = \frac{count(w)^{3/4}}{\sum_{w'} count(w')^{3/4}}$$

这样能提高低频词的采样概率。

### 5.2 编程题

做了什么操作： 定义了一个 cbow_forward 函数，实现 CBOW 模型的前向传播和损失计算。对每个样本，先获取上下文词的嵌入向量并取平均作为隐藏层，然后计算输出分数并用数值稳定的 softmax 得到概率分布，最后计算交叉熵损失。测试时随机初始化了嵌入矩阵和输出矩阵，打印了损失值。

In [6]:
import numpy as np

def cbow_forward(context_indices, target_idx, W, W_out):
    """
    context_indices: list of lists, each list contains context_size indices
    target_idx: list of target word indices
    W: (V, d) input embedding matrix
    W_out: (d, V) output embedding matrix
    返回: 平均交叉熵损失
    """
    batch_size = len(context_indices)
    losses = []
    
    for b in range(batch_size):
        # 获取上下文词的嵌入向量
        context_vecs = W[context_indices[b]]  # (context_size, d)
        
        # 计算平均上下文向量作为隐藏层
        h = np.mean(context_vecs, axis=0, keepdims=True)  # (1, d)
        
        # 计算输出分数和概率
        scores = np.dot(h, W_out)  # (1, V)
        # 数值稳定 softmax
        scores_shift = scores - np.max(scores)
        exp_scores = np.exp(scores_shift)
        probs = exp_scores / np.sum(exp_scores)  # (1, V)
        
        # 交叉熵损失
        loss = -np.log(probs[0, target_idx[b]] + 1e-10)
        losses.append(loss)
    
    return np.mean(losses)


# 测试
V, d, context_size, batch_size = 10, 4, 3, 2
W = np.random.randn(V, d)
W_out = np.random.randn(d, V)
context_indices = [[1, 2, 3], [4, 5, 6]]
target_idx = [0, 7]

loss = cbow_forward(context_indices, target_idx, W, W_out)
print("CBOW 损失:", loss)

CBOW 损失: 3.175066196034517


## 6 注意力机制

### 6.1 理论计算题

Q ∈ ℝ^(2×4)，K ∈ ℝ^(3×4)，V ∈ ℝ^(3×5)，d_k = 4

**Step 1: 计算得分矩阵**

$$\text{Scores} = \frac{QK^T}{\sqrt{d_k}} = \frac{QK^T}{2}$$

QK^T 为 2×3 矩阵，每个元素为 Q 的第 i 行与 K 的第 j 行的点积。

$$S_{ij} = \frac{1}{2} \sum_{m=1}^{4} Q_{im} K_{jm}$$

**Step 2: 对每行做 Softmax**

$$\text{Attention Weights}_{ij} = \frac{\exp(S_{ij})}{\sum_{k=1}^{3} \exp(S_{ik})}$$

得到 2×3 的注意力权重矩阵。

**Step 3: 加权求和得到输出**

$$\text{Output} = \text{Attention Weights} \times V$$

Output 形状为 2×5，其中第 i 行：

$$\text{Output}_i = \sum_{j=1}^{3} \text{Attention Weights}_{ij} \cdot V_j$$

### 6.2 编程题

做了什么操作： 定义了一个 MultiHeadAttention 类，实现多头注意力的前向传播。在初始化中定义了 Q、K、V 的线性投影层和最终的输出投影层。前向传播时先将输入 X 分别投影并拆分为 num_heads 个头部，对每个头计算缩放点积注意力，然后将所有头的输出拼接并通过最终线性层。测试时输入形状为 (8, 4, 4)，num_heads=2，打印了输入和输出形状。

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_v = d_model // num_heads
        
        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, X):
        seq_len, batch, d_model = X.shape
        
        # 线性投影并拆分为多头
        Q = self.W_q(X).view(seq_len, batch, self.num_heads, self.d_k).transpose(0, 2)
        K = self.W_k(X).view(seq_len, batch, self.num_heads, self.d_k).transpose(0, 2)
        V = self.W_v(X).view(seq_len, batch, self.num_heads, self.d_v).transpose(0, 2)
        # Q, K, V: (num_heads, seq_len, batch, d_k)
        
        # 缩放点积注意力
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        attn_weights = F.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, V)
        # attn_output: (num_heads, seq_len, batch, d_k)
        
        # 拼接多头
        attn_output = attn_output.transpose(0, 2).contiguous()
        attn_output = attn_output.view(seq_len, batch, d_model)
        
        # 最终线性层
        output = self.W_o(attn_output)
        
        return output


# 测试
seq_len, batch, d_model, num_heads = 8, 4, 4, 2
mha = MultiHeadAttention(d_model, num_heads)
X = torch.randn(seq_len, batch, d_model)

output = mha(X)
print("输入形状:", X.shape)
print("输出形状:", output.shape)

输入形状: torch.Size([8, 4, 4])
输出形状: torch.Size([8, 4, 4])
